In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import joblib
import os

In [3]:
df = pd.read_csv("../data/application_train.csv")

xgb_model = joblib.load("../outputs/xgboost_model.pkl")

In [4]:
y = df["TARGET"]

X = df.drop(columns=["TARGET", "SK_ID_CURR"])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
default_probability = xgb_model.predict_proba(X_test)[:, 1]

In [7]:
def probability_to_score(probability, min_score=300, max_score=850):
    score = max_score - probability * (max_score - min_score)
    return np.round(score).astype(int)

In [8]:
def assign_risk_tier(score):
    if score >= 750:
        return "Very Low Risk"
    elif score >= 700:
        return "Low Risk"
    elif score >= 650:
        return "Medium Risk"
    elif score >= 600:
        return "High Risk"
    else:
        return "Very High Risk"

In [9]:
def assign_risk_tier(score):
    if score >= 750:
        return "Very Low Risk"
    elif score >= 700:
        return "Low Risk"
    elif score >= 650:
        return "Medium Risk"
    elif score >= 600:
        return "High Risk"
    else:
        return "Very High Risk"

In [10]:
def base_credit_decision(score):
    """
    Basic credit decision based on credit score.
    """
    if score >= 700:
        return "Approve"
    elif score >= 600:
        return "Manual Review"
    else:
        return "Reject"

In [11]:
results = X_test.copy()

results["actual_default"] = y_test.values
results["default_probability"] = default_probability
results["credit_score"] = probability_to_score(default_probability)
results["risk_tier"] = results["credit_score"].apply(assign_risk_tier)
results["base_decision"] = results["credit_score"].apply(base_credit_decision)

results[[
    "actual_default",
    "default_probability",
    "credit_score",
    "risk_tier",
    "base_decision"
]].head(20)

,actual_default,default_probability,credit_score,risk_tier,base_decision
256571,0,0.382572,640,High Risk,Manual Review
191493,0,0.308149,681,Medium Risk,Manual Review
103497,0,0.793484,414,Very High Risk,Reject
130646,0,0.417316,620,High Risk,Manual Review
211898,0,0.476246,588,Very High Risk,Reject
130549,0,0.675196,479,Very High Risk,Reject
287975,0,0.161832,761,Very Low Risk,Approve
144906,0,0.110645,789,Very Low Risk,Approve
102173,0,0.834880,391,Very High Risk,Reject
254495,1,0.507426,571,Very High Risk,Reject


In [12]:
def probability_to_score(probability, min_score=300, max_score=850):
    score = max_score - probability * (max_score - min_score)
    return np.round(score).astype(int)


def assign_risk_tier(score):
    if score >= 750:
        return "Very Low Risk"
    elif score >= 700:
        return "Low Risk"
    elif score >= 650:
        return "Medium Risk"
    elif score >= 600:
        return "High Risk"
    else:
        return "Very High Risk"


def base_credit_decision(score):
    if score >= 700:
        return "Approve"
    elif score >= 600:
        return "Manual Review"
    else:
        return "Reject"

In [13]:
business_df = results.copy()

In [14]:
business_df["credit_to_income_ratio"] = (
    business_df["AMT_CREDIT"] / business_df["AMT_INCOME_TOTAL"]
)

business_df["annuity_to_income_ratio"] = (
    business_df["AMT_ANNUITY"] / business_df["AMT_INCOME_TOTAL"]
)

In [15]:
business_df[[
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "credit_to_income_ratio",
    "annuity_to_income_ratio"
]].head()

,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,credit_to_income_ratio,annuity_to_income_ratio
256571,157500.0,770292.0,30676.5,4.890743,0.194771
191493,90000.0,364896.0,19926.0,4.054400,0.221400
103497,148500.0,284400.0,18643.5,1.915152,0.125545
130646,188100.0,976711.5,38218.5,5.192512,0.203182
211898,180000.0,323194.5,19660.5,1.795525,0.109225


In [16]:
business_df[[
    "credit_to_income_ratio",
    "annuity_to_income_ratio"
]].describe()

,credit_to_income_ratio,annuity_to_income_ratio
count,61503.000000,61501.000000
mean,3.950268,0.181007
std,2.698247,0.094584
min,0.103741,0.009478
25%,2.019379,0.115300
50%,3.250000,0.162780
75%,5.134286,0.229174
max,84.736842,1.875965


In [17]:
def apply_business_rules(row):
    """
    This function applies heuristic business rules on top of the model-based score decision.
    """

    decision = row["base_decision"]
    reasons = []

    # Rule 1: Very low score is rejected directly
    if row["credit_score"] < 550:
        decision = "Reject"
        reasons.append("Credit score below 550")

    # Rule 2: High predicted default probability
    if row["default_probability"] >= 0.60:
        decision = "Reject"
        reasons.append("Default probability above 60%")

    # Rule 3: Excessive credit amount compared to income
    if row["credit_to_income_ratio"] > 8:
        if decision == "Approve":
            decision = "Manual Review"
        reasons.append("Credit amount is high relative to income")

    # Rule 4: High annuity burden compared to income
    if row["annuity_to_income_ratio"] > 0.40:
        if decision == "Approve":
            decision = "Manual Review"
        reasons.append("Annuity burden is high relative to income")

    # Rule 5: Very low external score
    if pd.notnull(row["EXT_SOURCE_2"]) and row["EXT_SOURCE_2"] < 0.20:
        if decision == "Approve":
            decision = "Manual Review"
        reasons.append("Low external risk score")

    # Rule 6: Short employment history
    if pd.notnull(row["DAYS_EMPLOYED"]):
        years_employed = abs(row["DAYS_EMPLOYED"]) / 365

        if years_employed < 1:
            if decision == "Approve":
                decision = "Manual Review"
            reasons.append("Employment history below 1 year")

    if len(reasons) == 0:
        reasons.append("No rule triggered")

    return pd.Series([decision, "; ".join(reasons)])

In [18]:
business_df[["final_decision", "decision_reasons"]] = business_df.apply(
    apply_business_rules,
    axis=1
)

In [19]:
business_df[[
    "actual_default",
    "default_probability",
    "credit_score",
    "risk_tier",
    "base_decision",
    "final_decision",
    "decision_reasons"
]].head(20)

,actual_default,default_probability,credit_score,risk_tier,base_decision,final_decision,decision_reasons
256571,0,0.382572,640,High Risk,Manual Review,Manual Review,Employment history below 1 year
191493,0,0.308149,681,Medium Risk,Manual Review,Manual Review,No rule triggered
103497,0,0.793484,414,Very High Risk,Reject,Reject,Credit score below 550; Default probability ab...
130646,0,0.417316,620,High Risk,Manual Review,Manual Review,Employment history below 1 year
211898,0,0.476246,588,Very High Risk,Reject,Reject,No rule triggered
130549,0,0.675196,479,Very High Risk,Reject,Reject,Credit score below 550; Default probability ab...
287975,0,0.161832,761,Very Low Risk,Approve,Approve,No rule triggered
144906,0,0.110645,789,Very Low Risk,Approve,Approve,No rule triggered
102173,0,0.834880,391,Very High Risk,Reject,Reject,Credit score below 550; Default probability ab...
254495,1,0.507426,571,Very High Risk,Reject,Reject,No rule triggered


In [20]:
business_df[["base_decision", "final_decision"]].value_counts()

base_decision  final_decision
Reject         Reject            24257
Manual Review  Manual Review     19015
Approve        Approve           15561
               Manual Review      2670
Name: count, dtype: int64

In [21]:
changed_decisions = business_df[
    business_df["base_decision"] != business_df["final_decision"]
]

changed_decisions.shape

(2670, 129)

In [22]:
changed_decisions[[
    "actual_default",
    "default_probability",
    "credit_score",
    "base_decision",
    "final_decision",
    "decision_reasons",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "credit_to_income_ratio",
    "annuity_to_income_ratio"
]].head(20)

,actual_default,default_probability,credit_score,base_decision,final_decision,decision_reasons,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,credit_to_income_ratio,annuity_to_income_ratio
187319,0,0.144967,770,Approve,Manual Review,Credit amount is high relative to income,180000.0,1546020.0,40914.0,8.589000,0.227300
23642,0,0.208995,735,Approve,Manual Review,Low external risk score,49500.0,119925.0,11812.5,2.422727,0.238636
276723,0,0.133845,776,Approve,Manual Review,Credit amount is high relative to income,135000.0,1647000.0,43578.0,12.200000,0.322800
198061,0,0.174431,754,Approve,Manual Review,Credit amount is high relative to income,112500.0,1546020.0,42642.0,13.742400,0.379040
266267,0,0.273489,700,Approve,Manual Review,Employment history below 1 year,202500.0,312840.0,8964.0,1.544889,0.044267
16264,0,0.188900,746,Approve,Manual Review,Credit amount is high relative to income,72000.0,675000.0,21906.0,9.375000,0.304250
180307,0,0.193425,744,Approve,Manual Review,Credit amount is high relative to income,103500.0,1017000.0,29866.5,9.826087,0.288565
36289,0,0.144633,770,Approve,Manual Review,Annuity burden is high relative to income,90000.0,538389.0,37602.0,5.982100,0.417800
86706,0,0.184895,748,Approve,Manual Review,Employment history below 1 year,261000.0,314100.0,15241.5,1.203448,0.058397
306803,0,0.101933,794,Approve,Manual Review,Credit amount is high relative to income,135000.0,1386265.5,36697.5,10.268633,0.271833


In [23]:
final_decision_analysis = business_df.groupby("final_decision").agg(
    applicants=("actual_default", "count"),
    defaults=("actual_default", "sum"),
    default_rate=("actual_default", "mean"),
    avg_credit_score=("credit_score", "mean"),
    avg_default_probability=("default_probability", "mean")
).reset_index()

final_decision_analysis

,final_decision,applicants,defaults,default_rate,avg_credit_score,avg_default_probability
0,Approve,15561,305,0.019600,747.697706,0.186007
1,Manual Review,21685,967,0.044593,662.496472,0.340919
2,Reject,24257,3693,0.152245,509.150183,0.619725


## Business rules engine results

The business rules layer adjusted the initial model-based decisions by moving some approved applicants into manual review when specific risk signals were detected.

After applying the rules, the approved group had a lower observed default rate, decreasing from approximately 2.02% to 1.96%. The manual review group also maintained an intermediate risk level, while the rejected group remained the highest-risk segment with a default rate above 15%.

The most common rule-based adjustments were related to short employment history, high annuity burden relative to income, high credit amount relative to income, low external risk scores, and high predicted default probability.

This confirms that the business rules layer adds an interpretable decision-making component on top of the machine learning model. Instead of relying only on the predicted probability, the engine can now produce a final credit decision and explain the business reasons behind that decision.

In [24]:
os.makedirs("../outputs", exist_ok=True)

business_df[[
    "actual_default",
    "default_probability",
    "credit_score",
    "risk_tier",
    "base_decision",
    "final_decision",
    "decision_reasons",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "credit_to_income_ratio",
    "annuity_to_income_ratio"
]].to_csv("../outputs/business_rules_results.csv", index=False)

In [25]:
os.path.exists("../outputs/business_rules_results.csv")

True